# PineSentry-Fire — 1-click Colab reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zxsa0716/pinesentry-fire/blob/main/colab.ipynb)

**Tanager Open Data Competition 2026 · Heedo Choi · Kookmin University**

This notebook reproduces the v1.9 submission package end-to-end. It walks through:

1. **Setup** — clone the repo and install dependencies (~3 min)
2. **View pre-computed examples** — see the headline figures and JSON outputs that ship with the repo (no compute)
3. **Reproduce key analyses** — run pytest + check core metrics (~1 min, no large downloads)
4. **(Optional) Full pipeline** — download EMIT + 임상도 + COP-DEM + dNBR perimeters and rebuild HSI v1 from scratch (~60–90 min, requires NASA earthaccess login)
5. **Inspect outputs** — show every figure inline

**License**: CC-BY-4.0 · **Pre-registration**: weights locked at git commit `c181cc2` (2026-04-29)

## 1 · Setup

In [ ]:
!git clone --depth 1 https://github.com/zxsa0716/pinesentry-fire.git
%cd pinesentry-fire
!pip -q install -r requirements.txt
!python -c "import rioxarray, geopandas, sklearn, statsmodels, prosail, xarray; print('OK')"

## 2 · View pre-computed examples (no compute)

The repo ships with `examples/` — every key figure and JSON output is pre-committed so you can see results immediately.

In [ ]:
from IPython.display import Image, display, Markdown
import json, pathlib

fig_dir = pathlib.Path('examples/figures')
tab_dir = pathlib.Path('examples/tables')
print(f'Figures: {len(list(fig_dir.glob("*.png"))) + len(list(fig_dir.glob("*.gif")))}')
print(f'Tables : {len(list(tab_dir.glob("*.json")))} JSON + 1 CSV')
print()
print('Headline 9-panel hero figure:')
display(Image('examples/figures/01_HERO_GRAND_9panel.png'))

In [ ]:
display(Image('examples/figures/03_HERO_roc_envelope_5site.png'))

In [ ]:
display(Image('examples/figures/02_HERO_methods_6panel.png'))

In [ ]:
from IPython.display import HTML
HTML('<img src="examples/figures/16_sancheong_temporal_T-1.5mo_animation.gif" width="720"/>')

In [ ]:
print('5-SITE BOOTSTRAP 95% CI:')
for site, d in json.load(open('examples/tables/bootstrap_5site_95CI.json')).items():
    print(f"  {site:>10}  AUC {d['auc_mean']:.3f}  CI [{d['auc_q025']:.3f}, {d['auc_q975']:.3f}]  lift@10% {d['lift_mean']:.2f}x")

print()
print('GEE SPATIAL LOGIT (R-INLA equivalent):')
for site, d in json.load(open('examples/tables/GEE_spatial_logit.json')).items():
    p = d['p']
    p_str = '<1e-6' if p < 1e-6 else f'{p:.4f}'
    print(f"  {site:>10}  OR {d['odds_ratio']:6.2f}  p {p_str}")

print()
print('CROSS-SITE WEIGHT TRANSFER (pre-registration defense):')
ct = json.load(open('examples/tables/cross_site_weight_transfer_OSF_defense.json'))
for r in ct['rows']:
    print(f"  {r['site']:>10}  AUC pre-reg {r['auc_OSF']:.3f}  AUC Uiseong-fit {r['auc_Uiseongfit']:.3f}  delta {r['delta']:+.3f}")

## 3 · Sanity check the package

In [ ]:
!PYTHONPATH=src python -m pytest tests/ -v

## 4 · (Optional) Full pipeline — produce HSI v1 from raw EMIT

**Downloads ~60 GB total.** Skip this section if you only want to see the pre-computed outputs.

Required: NASA Earthdata account (free) — register at https://urs.earthdata.nasa.gov.

In [ ]:
# Uncomment to run the full pipeline (requires Earthdata login):
# !python -c 'import earthaccess; earthaccess.login(persist=True)'
# !python scripts/run_all_downloads.py            # ~60 GB, 8 layers
# !python scripts/build_hsi_v0.py uiseong         # AUC 0.697
# !python scripts/build_feature_stack.py uiseong  # 10-band stack
# !python scripts/build_hsi_v1.py uiseong         # AUC 0.747
# !python scripts/make_grand_hero.py              # rebuild HERO_GRAND.png
print('Skipping full pipeline — see examples/ for pre-computed outputs.')

## 5 · Inspect every output

In [ ]:
from IPython.display import Image, Markdown, display
import pathlib
for f in sorted(pathlib.Path('examples/figures').glob('*.png')):
    display(Markdown(f'### `{f.name}`'))
    display(Image(str(f)))

## Where to go next

- **Reading guide**: `REVIEWER_GUIDE.md` (5 / 15 / full paths)
- **Anticipated questions**: `REVIEWER_FAQ.md`
- **Full numerical tables**: `TABLE.md` (22 tables)
- **Academic writeup**: `PAPER.md` (sections 4.1-4.21)
- **Browser dashboard**: open `REPORT.html` in any browser
- **Live demo**: HuggingFace Spaces — see `HUGGINGFACE_SPACES.md`
- **Streamlit local**: `streamlit run streamlit_app/app.py`

**License**: CC-BY-4.0 · **GitHub**: https://github.com/zxsa0716/pinesentry-fire